# Active and candidate promotion

Paradigm never trains the active reflex in place. This notebook measures whether a candidate changed too much on stable anchor states before promotion.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from paradigm import ActiveCandidateRegistry, BehaviorDriftGate, ReflexCompiler
from paradigm.synthetic import make_traces, make_xy

active = ReflexCompiler(random_state=3).fit(make_traces(3000, seed=3), name="active-v1")
candidate = ReflexCompiler(random_state=4).fit(make_traces(3500, seed=4), name="candidate-v2")
X_anchor, _ = make_xy(1200, seed=99)

gate = BehaviorDriftGate(max_disagreement=0.05, max_total_variation=0.08)
report = gate.compare(active, candidate, X_anchor)
print(report)

registry = ActiveCandidateRegistry(active=active)
registry.stage(candidate)
if report.accepted:
    registry.promote()
else:
    registry.reject("behavior drift gate")
print(registry.history)

## Important distinction

This output-level drift test is not the Drift Contract optimizer. The later bounded-plasticity experiment should implement the spectral update rule from [drift-contract](https://github.com/infinition/drift-contract) and compare activation drift with behavior drift.